# Generative Sequential Recommendation — Colab 训练 (T5 + Semantic ID)

**运行前**：Runtime → Change runtime type → **GPU**（推荐 A100）

**架构**：T5 encoder-decoder（对齐 TIGER 参考实现），从头训练，~4.6M 参数。
历史序列拍平成 Semantic ID token 输入 encoder，decoder 自回归生成下一个 item 的 4 个 token。

**需要手动上传**：
- `embedding/semantic_ids_rqvae.npy`（main 线，best ckpt 出的 SID）
- 或 `embedding/semantic_ids_rqvae_final.npy`（E14 ablation，final ckpt 出的 SID）
- 或任何其他变体（random ID、不同 prompt 等），改 Cell 4 顶部 `SID_FILE` 即可

`data/beauty_data.pkl` 已在 GitHub 仓库中，clone 后自动存在。

本 notebook 只跑下游 T5 训练 + 评估，不重新训 RQ-VAE。
如需重训 RQ-VAE 走完整 pipeline，参考 `notebooks/colab_full_pipeline.ipynb`。

---

## 运行顺序
1. **环境**（Cell 1–3）：GPU 检查 → 装依赖 → clone repo
2. **配置 + 上传**（Cell 4）：在 Cell 4 顶部设置 `SID_FILE` / `CKPT_FILE`，然后上传对应的 SID 文件
3. **训练 + 评估**（Cell 5–6）：`train.py` → `evaluate.py`（自动用 Cell 4 的变量）
4. **下载结果**（Cell 7）

## E14 best vs final A/B 用法

跑两条线：
- 第一次跑 best：`SID_FILE='semantic_ids_rqvae.npy'`，`CKPT_FILE='checkpoints/best_model_t5_e14_best.pt'`
- 第二次跑 final（开新 session）：`SID_FILE='semantic_ids_rqvae_final.npy'`，`CKPT_FILE='checkpoints/best_model_t5_e14_final.pt'`

In [ ]:
# Cell 1：确认 GPU 可用
import torch
print(f'GPU 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 型号: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  未检测到 GPU，请先到 Runtime → Change runtime type → GPU')

In [ ]:
# Cell 2：安装依赖
!pip install transformers scikit-learn -q
print('依赖安装完成 ✅')

In [ ]:
# Cell 3：clone 仓库（或 pull 更新）
import os
REPO = 'Generative-Sequential-Recommendation'
if not os.path.exists(REPO):
    !git clone https://github.com/rhine-yangrui/Generative-Sequential-Recommendation.git
else:
    print('repo 已存在，执行 git pull 同步最新提交')
    !cd {REPO} && git pull
%cd {REPO}
!git log --oneline -3

In [ ]:
# Cell 4：选择本次跑哪条线 + 上传对应 SID 文件
import os
from google.colab import files

os.makedirs('embedding', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

# ─────────────────────────────────────────────────────────────────────
# 改这两行就能切换 best vs final 变体（或任何其他 SID 文件）
# Cell 5/6/7 会通过 shell 替换自动用这两个变量，不要在下面写死路径。
# ─────────────────────────────────────────────────────────────────────
SID_FILE  = 'semantic_ids_rqvae.npy'                  # E14 best ckpt 出的 SID
CKPT_FILE = 'checkpoints/best_model_t5_e14_best.pt'   # 训练后的输出 ckpt

# 跑 final 变体时改成：
# SID_FILE  = 'semantic_ids_rqvae_final.npy'
# CKPT_FILE = 'checkpoints/best_model_t5_e14_final.pt'

print(f'本次跑：')
print(f'  SID:  embedding/{SID_FILE}')
print(f'  Ckpt: {CKPT_FILE}')
print(f'\n请上传与上面 SID_FILE 对应的本地 .npy 文件')
uploaded = files.upload()
for fname in uploaded:
    dest = f'embedding/{fname}'
    os.rename(fname, dest)
    print(f'✅ 已移动到 {dest}  ({os.path.getsize(dest)/1024:.0f} KB)')

# Sanity check：用 SID_FILE 验证刚刚上传的那份
import numpy as np
sids = np.load(f'embedding/{SID_FILE}', allow_pickle=True).item()
vals = list(sids.values())
print(f'\n✅ 加载成功：{len(sids)} items')
print(f'   示例：item {list(sids.keys())[0]} → {vals[0]}')
assert len(vals[0]) == 4, f'期望 4-token ID，实际 {len(vals[0])}'

ranges = list(zip(*vals))
for i, r in enumerate(ranges):
    print(f'   c{i} 范围: {min(r)}..{max(r)}  usage: {len(set(r))}')

unique_ids = len({tuple(v) for v in vals})
print(f'   唯一 4-tuple 数: {unique_ids} / {len(sids)}  '
      f'{"✓ 零冲突" if unique_ids == len(sids) else "✗ 有冲突！"}')

---
## 训练生成式推荐模型（T5 encoder-decoder + RQ-VAE 4-token Semantic ID）

In [ ]:
# Cell 5：训练 T5
# 200 epoch，全量 val 每 2 epoch 一次，patience=10 早停（主线通常 ~130 epoch 触发）
# A100 实际耗时 ~2-3 小时；T4 显存不够会 OOM，需要把 train.py CONFIG['batch_size'] 降到 256
# 常数 lr=1e-4 + Adam 无 weight decay，对齐 TIGER 参考实现（无 scheduler）
# 早停依据 validation NDCG@10
# 输出 ckpt 路径来自 Cell 4 的 CKPT_FILE 变量
!python train.py --semantic-ids {SID_FILE} --ckpt {CKPT_FILE}

In [ ]:
# Cell 6：评估（all-rank Recall@K / NDCG@K，beam=50 constrained beam search）
!python evaluate.py --semantic-ids {SID_FILE} --ckpt {CKPT_FILE}

---
## 下载结果

In [ ]:
# Cell 7：下载 checkpoint（路径来自 Cell 4 的 CKPT_FILE）
import os
from google.colab import files

if os.path.exists(CKPT_FILE):
    files.download(CKPT_FILE)
    print(f'✅ 下载：{CKPT_FILE}  ({os.path.getsize(CKPT_FILE)/1e6:.1f} MB)')
else:
    print(f'⚠️  {CKPT_FILE} 不存在')